# 第2章 プロンプト設計 実践

この Notebook では、プロンプト設計を「文章を読む」だけでなく、ブラウザ Chat UI で実際に試し、Python APIで再現し、Agentic AIでファイル文脈を読ませるところまで体験します。

In [ ]:
from pathlib import Path
import os
import sys

# Notebook をどこから開いても helper を import できるようにします。
search_roots = [Path.cwd()]
env_root = os.environ.get("LOCAL_LLM_REPO_ROOT")
if env_root:
    search_roots.append(Path(env_root))
search_roots.append(Path("C:/LLM"))

seen = set()
for root in search_roots:
    current = root.resolve()
    for candidate in [current, *current.parents]:
        if candidate in seen:
            continue
        seen.add(candidate)
        helper_dir = candidate / "notebooks"
        if (helper_dir / "local_llm_practice.py").exists():
            sys.path.insert(0, str(helper_dir))
            break
    else:
        continue
    break
else:
    raise RuntimeError(
        "notebooks/local_llm_practice.py が見つかりません。"
        "C:/LLM か notebooks/ 配下で開くか、LOCAL_LLM_REPO_ROOT を設定してください。"
    )

from local_llm_practice import (
    configure_local_caches,
    copy_to_clipboard,
    load_chapter,
    ollama_generate,
    open_aider_terminal,
    prepare_aider_practice_workspace,
    print_headings,
    REPO_ROOT,
    DOCS_DIR,
    WORK_DIR,
    start_ollama_chat_ui,
    stop_ollama_chat_ui,
)

configure_local_caches()
print("REPO_ROOT:", REPO_ROOT)
print("DOCS_DIR :", DOCS_DIR)
print("WORK_DIR :", WORK_DIR)

chapter_path, chapter_text = load_chapter("02-prompt-design.md")
print(chapter_path)
print_headings(chapter_text)


## 1. Chat UI を開いて、短い依頼を試す

次のセルでブラウザ Chat UI を開きます。開いた画面で、まず短い依頼を入力します。

```text
ローカルLLMの始め方を教えて。
```

返答を見たら、次に条件を足して続けます。

```text
読者は RTX 4060 Ti 16GB のPCでOllamaを使い始めたところです。今日やることを3つ、まだやらないことを2つに分けてください。
```

ここで体験するのは、Chat UI では会話しながら条件を追加できる、ということです。

In [ ]:
short_request = "ローカルLLMの始め方を教えて。"
start_ollama_chat_ui()
copy_to_clipboard(short_request)


## 2. Python API で同じ違いを再現する

Chat UI で変化を見た後、Notebook で同じ2つの依頼を実行します。Python API は、同じ条件で比較したい時に向いています。

In [ ]:
structured_request = """
読者は RTX 4060 Ti 16GB のPCでOllamaを使い始めたところです。
今日やることを3つ、まだやらないことを2つに分けてください。
各項目は、実行できる確認作業として書いてください。
""".strip()

for title, request in [("短い依頼", short_request), ("条件を足した依頼", structured_request)]:
    print(f"\n=== {title} ===")
    print(ollama_generate(request, temperature=0.2))


## 3. Agentic AI で「ファイルを読ませる」体験をする

次は aider を開き、Notebook や README を読ませます。開いた PowerShell で次を順に入力します。

1. `/help`
2. `/add README.md`
3. `/add notebooks/local-llm-customization/02-prompt-design.ipynb`
4. `このNotebookを初めて読む人が、プロンプト設計で何を体験するのか説明してください。まだ編集しないでください。`
5. `/exit`

ここでの目的は、Agentic AI がファイル文脈を持って会話できることを体験することです。
実リポジトリを誤って編集しないように、この章では `work/aider-practice/` に作る練習用 workspace を開きます。ここは `.gitignore` されているため、教材の tracked files は変更されません。


In [ ]:
practice_path = prepare_aider_practice_workspace()
open_aider_terminal(practice_path)


## 結果の読み方 / 次へ進む判断

- 短い依頼より、条件を足した依頼の方が「今日やること」「まだやらないこと」の形にそろっていれば、プロンプト設計の効果を確認できています。
- Chat UI では追加質問で条件を足し、Python API では同じ依頼を再実行できることを見ます。
- aider で Notebook や README の文脈を読ませられたら、プロンプトに直接貼らない文脈の渡し方を体験できています。

次は第3章で、根拠文書を検索してから回答へ渡す RAG に進みます。

Chat UI を使い終わったら、同じ kernel で `stop_ollama_chat_ui()` を実行すると教材用サーバーを終了できます。Notebook の kernel を再起動しても終了します。
